# Group 2 — Build Our Own De-Identification Pipeline
**Supervisor decision (after the Oct 12 meeting): Group 2 builds its own de-identified data,
rather than waiting on Group 1.**

This notebook produces a de-identified version of the dataset, so you can re-run your existing
`group2_model_training_pipeline.ipynb` on it and compare raw-vs-de-identified performance,
which is the project's core question.

## Design, and one important tension to flag up front
Two different kinds of column need different treatment:

- **Quasi-identifiers** — fields that could help re-identify someone if combined with outside
  information: `sex`, `bmi`, `weight`, `height`. These get **generalised** (binned into coarser
  categories) and checked for **k-anonymity**.
- **Sensitive prediction targets** — `age` and `saturation`. These are exactly what the project
  is trying to predict, so they are **not** generalised or altered. Instead, they are kept in a
  **separate labels file**, joined back only via a pseudonym, not stored alongside the
  quasi-identifiers. This mirrors a standard data-governance pattern: never release the sensitive
  attribute and the re-identifying attributes in the same table.

**Note on age specifically:** age is unusual here because it is *both* a quasi-identifier in the
general privacy literature *and* this project's prediction target. We resolve this by treating it
purely as a label (exact, in the restricted file) rather than generalising it — generalising your
own prediction target would make the whole exercise meaningless. If your supervisors want age
itself treated as a quasi-identifier too, that is a policy call to raise separately.

**Before running:** set `DATA_DIR` and `K_THRESHOLD` below. Defaults are reasonable but not yet
confirmed by your supervisors (see the chat discussion) — update them once you hear back.


In [ ]:
import json
import datetime
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import find_peaks

warnings.filterwarnings("ignore")

# ---- CONFIG ----
DATA_DIR = Path(r".")   # e.g. Path(r"C:\Users\<you>\mcd_rppg")
DB_CSV = DATA_DIR / "db.csv"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

K_THRESHOLD = 5          # minimum group size for k-anonymity — confirm with supervisors
QUASI_IDENTIFIERS = ["sex", "bmi", "weight", "height"]   # confirm scope with supervisors
SENSITIVE_LABELS = ["age", "saturation"]                  # prediction targets — kept separate

PSEUDONYM_SALT = "group2-mcd-rppg-2026"   # keep this constant secret; do not share it

assert DB_CSV.exists(), f"Could not find {DB_CSV} — update DATA_DIR above."
print("Using data dir:", DATA_DIR.resolve())
print(f"k-anonymity threshold: {K_THRESHOLD}")
print(f"Quasi-identifiers to generalise: {QUASI_IDENTIFIERS}")
print(f"Sensitive labels (kept exact, in a separate file): {SENSITIVE_LABELS}")


## 1. Load data and extract the same features as the training pipeline
Self-contained: re-extracts HRV/SDPPG features directly from the local PPG files, so this
notebook doesn't depend on anything being saved from the earlier training notebook.


In [ ]:
df = pd.read_csv(DB_CSV)
subj = df.drop_duplicates(["patient_id", "step"]).reset_index(drop=True)
print("Subject-state rows:", len(subj))


def load_pw(path):
    vals, times = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            vals.append(float(parts[0]))
            times.append(parts[1])
    return np.array(vals, dtype=float), times


def ppg_sampling_rate(times):
    fmt = "%Y-%m-%d %H:%M:%S.%f"
    t0 = datetime.datetime.strptime(times[0], fmt)
    t1 = datetime.datetime.strptime(times[-1], fmt)
    duration = (t1 - t0).total_seconds()
    return (len(times) / duration if duration > 0 else None), duration


def hrv_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi >= 60/max_bpm) & (ibi <= 60/min_bpm)]
    if len(ibi) < 3:
        return None
    hr = 60 / ibi
    nn_diff = np.diff(ibi) * 1000
    pnn50 = np.mean(np.abs(nn_diff) > 50) * 100 if len(nn_diff) else np.nan
    return {
        "mean_hr": hr.mean(), "hr_std": hr.std(), "hr_range": hr.max() - hr.min(),
        "mean_ibi": ibi.mean(), "sdnn": ibi.std() * 1000,
        "rmssd": np.sqrt(np.mean(nn_diff**2)) if len(nn_diff) else np.nan,
        "pnn50": pnn50, "n_beats": len(peaks),
    }


def sdppg_features(signal, fs, min_bpm=40, max_bpm=180):
    '''Simplified/heuristic SDPPG a/b-wave features — not a validated fiducial detector.'''
    if fs is None or len(signal) < 10:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    d2 = np.diff(signal, n=2)
    a_vals, b_vals = [], []
    for i in range(len(peaks) - 1):
        seg = d2[peaks[i]:peaks[i + 1]]
        if len(seg) < 5:
            continue
        half = seg[: max(len(seg) // 2, 3)]
        a, b = half.max(), half.min()
        if a > 0:
            a_vals.append(a)
            b_vals.append(b)
    if len(a_vals) < 3:
        return None
    a_vals, b_vals = np.array(a_vals), np.array(b_vals)
    return {"sdppg_a_mean": a_vals.mean(), "sdppg_b_mean": b_vals.mean(),
            "sdppg_ba_ratio": (b_vals / a_vals).mean(), "sdppg_n_beats": len(a_vals)}


In [ ]:
rows = []
for _, r in subj.iterrows():
    stem = f"{r.patient_id}_{r.step}"
    pw_path = DATA_DIR / r.ppg
    row = {
        "patient_id": r.patient_id, "step": r.step,
        "sex": r.sex, "bmi": r.bmi, "weight": r.weight, "height": r.height,
        "age": r.age, "saturation": r.saturation,
    }
    if pw_path.exists():
        try:
            vals, times = load_pw(pw_path)
            fs, dur = ppg_sampling_rate(times)
            hrv = hrv_features(vals, fs)
            sdp = sdppg_features(vals, fs)
            if hrv:
                row.update(hrv)
            if sdp:
                row.update(sdp)
            row["ppg_found"] = True
        except Exception as e:
            print(f"  skipping {pw_path.name}: {e}")
            row["ppg_found"] = False
    else:
        row["ppg_found"] = False
    rows.append(row)

raw_table = pd.DataFrame(rows)
FEATURE_COLS = ["mean_hr", "hr_std", "hr_range", "mean_ibi", "sdnn", "rmssd", "pnn50",
                 "sdppg_a_mean", "sdppg_b_mean", "sdppg_ba_ratio"]
n_usable = raw_table.dropna(subset=FEATURE_COLS).shape[0]
print(f"PPG files found locally: {raw_table.ppg_found.sum()} / {len(raw_table)}")
print(f"Rows with a complete feature set: {n_usable} / {len(raw_table)}")


## 2. Pseudonymise `patient_id`
Replace the real subject ID with a one-way hash, so the released tables carry no reversible
identifier. The mapping back to the real ID is written to a **local-only** crosswalk file —
never share this file.


In [ ]:
def pseudonymise(patient_id, salt=PSEUDONYM_SALT):
    raw = f"{salt}:{patient_id}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]   # 16 hex chars is plenty for this dataset size


raw_table["pseudonym_id"] = raw_table["patient_id"].apply(pseudonymise)

# Sanity check: pseudonymisation must be 1-to-1 (no collisions) and consistent per subject
n_real = raw_table["patient_id"].nunique()
n_pseudo = raw_table["pseudonym_id"].nunique()
print(f"Real subject IDs: {n_real}  Pseudonym IDs: {n_pseudo}  (should match, no collisions)")
assert n_real == n_pseudo, "Pseudonym collision detected — widen the hash length."

crosswalk = raw_table[["patient_id", "pseudonym_id"]].drop_duplicates()
crosswalk_path = RESULTS_DIR / "crosswalk_LOCAL_ONLY_DO_NOT_SHARE.csv"
crosswalk.to_csv(crosswalk_path, index=False)
print(f"Saved local-only crosswalk to: {crosswalk_path}")
print("This file must NOT be shared, uploaded, or committed alongside the de-identified data.")


## 3. Generalise the quasi-identifiers
- `bmi` → standard WHO categories (underweight / normal / overweight / obese)
- `weight`, `height` → equal-width bins (coarse enough to avoid being near-unique)
- `sex` → kept as-is (already a 2-category field; further generalising it to "any" would remove
  it entirely, which is a suppression decision, not a binning one — see Section 4 if needed)


In [ ]:
def bmi_category(bmi):
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return "underweight"
    if bmi < 25:
        return "normal"
    if bmi < 30:
        return "overweight"
    return "obese"


def width_bin(series, width):
    '''Equal-width binning starting from the series minimum.'''
    lo = np.floor(series.min() / width) * width
    bins = np.arange(lo, series.max() + width, width)
    labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins) - 1)]
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True)


deid = raw_table.copy()
deid["bmi_bin"] = deid["bmi"].apply(bmi_category)
deid["weight_bin"] = width_bin(deid["weight"], width=10)     # 10 kg bins, widened later if needed
deid["height_bin"] = width_bin(deid["height"], width=10)     # 10 cm bins, widened later if needed
# sex is left as-is for now

print(deid[["bmi", "bmi_bin", "weight", "weight_bin", "height", "height_bin"]].head())


## 4. k-anonymity check, with automatic bin-widening
A group of (sex, bmi_bin, weight_bin, height_bin) is **not k-anonymous** if fewer than
`K_THRESHOLD` subjects share that exact combination. This widens the weight/height bins
automatically until every group meets the threshold, or suppresses `height_bin`/`weight_bin`
entirely as a last resort if widening alone isn't enough.


In [ ]:
def smallest_group_size(data, qi_cols):
    sizes = data.drop_duplicates("patient_id").groupby(qi_cols, observed=True)["patient_id"].nunique()
    return sizes.min(), sizes


def check_k_anonymity(data, qi_cols, k):
    min_size, sizes = smallest_group_size(data, qi_cols)
    n_violating_groups = (sizes < k).sum()
    n_violating_subjects = sizes[sizes < k].sum()
    return min_size, n_violating_groups, int(n_violating_subjects)


qi_bin_cols = ["sex", "bmi_bin", "weight_bin", "height_bin"]
min_size, n_bad_groups, n_bad_subj = check_k_anonymity(deid, qi_bin_cols, K_THRESHOLD)
print(f"Initial check — smallest group: {min_size}, "
      f"groups below k={K_THRESHOLD}: {n_bad_groups} (covering {n_bad_subj} subjects)")

# Widen weight/height bins progressively if needed
for width in [20, 30, 50]:
    if n_bad_groups == 0:
        break
    print(f"\nWidening weight/height bins to {width} units and re-checking...")
    deid["weight_bin"] = width_bin(deid["weight"], width=width)
    deid["height_bin"] = width_bin(deid["height"], width=width)
    min_size, n_bad_groups, n_bad_subj = check_k_anonymity(deid, qi_bin_cols, K_THRESHOLD)
    print(f"  smallest group: {min_size}, groups below k: {n_bad_groups} ({n_bad_subj} subjects)")

if n_bad_groups > 0:
    print(f"\nStill {n_bad_groups} groups below k={K_THRESHOLD} after widening. "
          f"Dropping height_bin from the released quasi-identifiers.")
    qi_bin_cols = ["sex", "bmi_bin", "weight_bin"]
    min_size, n_bad_groups, n_bad_subj = check_k_anonymity(deid, qi_bin_cols, K_THRESHOLD)
    print(f"  smallest group: {min_size}, groups below k: {n_bad_groups} ({n_bad_subj} subjects)")

print(f"\nAfter widening + dropping columns, groups still below k: {n_bad_groups}")
print(f"k-anonymity satisfied by generalisation alone: {n_bad_groups == 0}")


## 4b. Cell suppression for any remaining violations
Widening and dropping whole columns doesn't always finish the job — rare demographic
combinations (e.g. one unusually tall, light subject) can stay below the threshold regardless of
bin width. The standard fix is **cell suppression**: for the specific rows whose group is still
too small, replace their quasi-identifier values with `"suppressed"` rather than revealing a
near-unique combination. This affects only the quasi-identifier columns — the HRV/SDPPG
features and the separate labels file are untouched, so model training is unaffected.


In [ ]:
def suppress_small_groups(data, qi_cols, k, max_rounds=5):
    data = data.copy()
    # pd.cut produces Categorical columns, which reject a new string value like
    # "suppressed" unless cast to a plain object dtype first.
    for col in qi_cols:
        data[col] = data[col].astype(object)
    for _ in range(max_rounds):
        sizes = (data.drop_duplicates("patient_id")
                      .groupby(qi_cols, observed=True)["patient_id"].nunique())
        bad_combos = sizes[sizes < k].index
        if len(bad_combos) == 0:
            break
        bad_combos_set = set(bad_combos) if len(qi_cols) > 1 else {(c,) for c in bad_combos}
        mask = data[qi_cols].apply(tuple, axis=1).isin(bad_combos_set)
        for col in qi_cols:
            data.loc[mask, col] = "suppressed"
    return data


deid = suppress_small_groups(deid, qi_bin_cols, K_THRESHOLD)
min_size, n_bad_groups, n_bad_subj = check_k_anonymity(deid, qi_bin_cols, K_THRESHOLD)
n_suppressed = (deid["bmi_bin"] == "suppressed").sum() if "bmi_bin" in qi_bin_cols else 0
print(f"After suppression — smallest group: {min_size}, groups below k: {n_bad_groups}")
print(f"Rows with suppressed quasi-identifiers: {n_suppressed} / {len(deid)}")
print(f"k-anonymity satisfied (k={K_THRESHOLD}): {n_bad_groups == 0}")
if n_bad_groups > 0:
    print("\nStill not satisfied — the 'suppressed' bucket itself is below k. This means the")
    print("dataset has more unique demographic outliers than k allows; raise with supervisors")
    print("whether to lower k, broaden the quasi-identifier bins further, or drop those subjects.")


## 5. Build the two output files
- **`features_deidentified.csv`** — pseudonym ID, generalised quasi-identifiers, and the
  HRV/SDPPG features (derived numbers, not raw waveforms). Safe to share within the team.
- **`labels_restricted.csv`** — pseudonym ID and the exact prediction targets (`age`,
  `saturation`). Kept separate; only join the two files back together at model-training time,
  never store them combined on disk.


In [ ]:
feature_output_cols = ["pseudonym_id", "step"] + qi_bin_cols + FEATURE_COLS
features_deidentified = deid[feature_output_cols].copy()

label_output_cols = ["pseudonym_id", "step"] + SENSITIVE_LABELS
labels_restricted = deid[label_output_cols].copy()

feat_path = RESULTS_DIR / "features_deidentified.csv"
label_path = RESULTS_DIR / "labels_restricted.csv"
features_deidentified.to_csv(feat_path, index=False)
labels_restricted.to_csv(label_path, index=False)

print(f"Saved: {feat_path}  ({len(features_deidentified)} rows)")
print(f"Saved: {label_path}  ({len(labels_restricted)} rows)")
print()
print("features_deidentified.csv preview:")
print(features_deidentified.head())
print()
print("labels_restricted.csv preview:")
print(labels_restricted.head())


## 6. Sanity checks before trusting this output


In [ ]:
print("CHECK 1 — no raw patient_id in either released file:")
leaked = ("patient_id" in features_deidentified.columns) or ("patient_id" in labels_restricted.columns)
print(f"  patient_id present in released files: {leaked}  (must be False)")

print()
print("CHECK 2 — no exact age/sex/bmi/weight/height leaking into the labels file beyond the")
print("          intended sensitive columns:")
unexpected = set(labels_restricted.columns) - {"pseudonym_id", "step"} - set(SENSITIVE_LABELS)
print(f"  unexpected columns in labels file: {unexpected}  (should be empty set)")

print()
print("CHECK 3 — row counts match the original subject-state table:")
print(f"  original: {len(deid)}, features file: {len(features_deidentified)}, "
      f"labels file: {len(labels_restricted)}")

print()
print("CHECK 4 — re-confirm k-anonymity on the exact file being released:")
min_size, n_bad_groups, n_bad_subj = check_k_anonymity(deid, qi_bin_cols, K_THRESHOLD)
print(f"  smallest released group size: {min_size} (threshold k={K_THRESHOLD})")
print(f"  groups still below threshold: {n_bad_groups}")

print()
print("CHECK 5 — pseudonym is consistent for the same subject across both states:")
check = deid.groupby("patient_id")["pseudonym_id"].nunique()
print(f"  subjects with more than 1 pseudonym (should be 0): {(check > 1).sum()}")


## 7. Next step: feed this into the model-training pipeline
To test the project's core question (does de-sensitised data retain model performance), load
these two files back into `group2_model_training_pipeline.ipynb` in place of the raw feature
extraction step, joining on `pseudonym_id` + `step`:

```python
features_raw = pd.read_csv(RESULTS_DIR / "features_deidentified.csv")
labels = pd.read_csv(RESULTS_DIR / "labels_restricted.csv")
features = features_raw.merge(labels, on=["pseudonym_id", "step"])
features = features.rename(columns={"pseudonym_id": "patient_id"})  # so existing code still works
features["sex_num"] = (features["sex"] == "M").astype(int)
```

Then re-run the same cross-validation and held-out test cells unchanged, and compare the
resulting MAEs against this week's raw-data results (age: 2.77 / 11.60 years by subgroup;
saturation: 0.91%).


In [ ]:
print("=" * 60)
print("SUMMARY CHECKLIST")
print("=" * 60)
checklist = [
    "[ ] Confirmed k threshold and quasi-identifier scope with supervisors",
    "[ ] Crosswalk file confirmed local-only and NOT shared with the team",
    "[ ] k-anonymity satisfied on the final released quasi-identifier columns",
    "[ ] features_deidentified.csv and labels_restricted.csv both saved",
    "[ ] Re-ran the model-training pipeline on the de-identified data",
    "[ ] Built the raw-vs-de-identified comparison table for age and saturation",
]
for item in checklist:
    print(item)
